# Getting started — Infant Gut Shotgun-Metagenome Catalog
Load the wide table, filter by confidence and route, join runs, and plot coverage. Files are expected in the same folder as this notebook.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
wide = pd.read_parquet('sample_metadata_wide.parquet')
studies = pd.read_parquet('study_metadata_wide.parquet')
print(wide.shape, studies.shape)
wide.head(3).T.head(30)

(153701, 82) (389, 71)


,0,1,2
sample_key,SAMN20560764,SAMN11811759,SAMN11811807
study_accession,PRJNA751712,PRJNA544346,PRJNA544346
secondary_sample,SRS9702382,SRS5723481,SRS5723479
sample_title,MIMARKS Survey related sample from human gut metagenome,mother_A8,infant_B28
body_site_class,primary,linked,primary
collection_date,2017-01-01/2021-02-19,2018-03-18,2018-01-19
is_gold_heldout,False,False,False
probiotic_exposure,NaN,NaN,NaN
preterm_status,preterm,NaN,NaN
gestational_age_weeks,NaN,NaN,NaN


## 1. Infant stool samples with a per-sample age (R1/R2 evidence)

In [ ]:
infant = wide[(wide.body_site_class.isin(['primary','unknown'])) & (~wide.adult_age_flag)]
aged = infant[infant.age_at_collection_days.notna() & infant['age_at_collection_days__route'].isin(['R1','R2'])]
print(len(infant), 'infant-scope samples;', len(aged), 'with per-sample age from archive/table evidence')
aged.age_at_collection_days.clip(upper=1100).plot.hist(bins=60, figsize=(6,3)); plt.xlabel('age at collection (days)'); plt.show()

137997 infant-scope samples; 48002 with per-sample age from archive/table evidence


## 2. A study-level view: which cohorts have delivery mode AND feeding for most samples?

In [ ]:
good = studies[(studies.cov_delivery_mode>=0.8) & (studies.cov_feeding_mode>=0.8)][['study_accession','study_title','n_samples','cov_age_at_collection_days','cov_delivery_mode','cov_feeding_mode','cohort_name']]
good.sort_values('n_samples', ascending=False).head(15)

,study_accession,study_title,n_samples,cov_age_at_collection_days,cov_delivery_mode,cov_feeding_mode,cohort_name
56,PRJEB32631,Early_life_microbiota_colonisation,1679,0.998,1.000,0.808,Shao 2019 UK
334,PRJEB62690,"Infant stool, 5-HMO supplemented formula vs. control formula vs. breastfed",1110,1.000,1.000,1.000,singleton study PRJEB62690
118,PRJEB10914,Developmental Ecology of the Human Microbiome - Restoration Study of babies.,1016,1.000,1.000,1.000,singleton study PRJEB10914
88,PRJNA1404625,Investigating the role of the infant gut microbiome in rotavirus vaccine efficacy,835,1.000,0.990,0.928,singleton study PRJNA1404625
136,PRJEB39610,Longitudinal metagenomic analysis of the gut microbiome in preterm infants diagnosed with necrotising enterocolitis and matched controls,644,1.000,1.000,0.972,NaN
295,PRJNA807448,The Antibiotics and Immune Responses Study,459,1.000,1.000,0.993,Ryan 2025 Australia
253,PRJNA489090,"Human infant gut metagenome, infant shotgun sequencing",429,1.000,1.000,0.928,singleton study PRJNA489090
250,PRJNA473126,Infant Diet and Maternal Gestational Weight Gain Influence Functional Maturation of the Infant Gut Microbiome,402,1.000,1.000,1.000,Baumann-Dudenhoeffer 2018 USA
244,PRJNA396794,preterm infant gut metagenomes (NIH Y4 Cohort),305,1.000,0.997,0.993,NIH Y4 Cohort
260,PRJNA549787,"South African HIV-exposed, uninfected infant gut microbiomes",165,1.000,0.994,1.000,D'Souza 2020 South Africa


## 3. Build an analysis slice and export it
Example: term, vaginally born, exclusively breastfed infants sampled before 6 months, with confidence ≥0.7 on every field used.

In [ ]:
sl = infant[(infant.preterm_status=='term') & (infant.delivery_mode=='vaginal') & (infant.feeding_mode=='exclusive_breast') & (infant.age_at_collection_days<183)]
for f in ['preterm_status','delivery_mode','feeding_mode','age_at_collection_days']:
    sl = sl[sl[f+'__confidence']>=0.7]
print(len(sl), 'samples in', sl.study_accession.nunique(), 'studies')
sl[['sample_key','study_accession','run_accessions','age_at_collection_days','delivery_mode','feeding_mode']].to_csv('my_slice.csv', index=False)

1160 samples in 15 studies


## 4. Trace a value back to its evidence

In [ ]:
det = pd.read_parquet('sample_determinations.parquet')
k = sl.sample_key.iloc[0]
det[det.sample_key==k][['field_name','value_normalized','confidence','route','evidence_source','evidence_locator','evidence_quote']]

,field_name,value_normalized,confidence,route,evidence_source,evidence_locator,evidence_quote
61927,age_at_collection_days,30,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!timepoint_months:135,timepoint_months=1.0
61928,birth_weight_grams,3600,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!birth_weight (Kg):135,birth_weight (Kg)=3.6
61929,country,ES,0.90,R1,biosample_attr,country,Spain
61930,delivery_mode,vaginal,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!delivery:135,delivery=homebirth
61931,feeding_mode,exclusive_breast,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!feeding_cat:135,feeding_cat=Breastfeeding
61932,gestational_age_weeks,40.0,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!gestational_age:135,gestational_age=40.0
61933,maternal_antibiotics,no,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!intrapartum_ATB:135,intrapartum_ATB=0.0
61934,multiple_birth,singleton,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!Twin:135,Twin=0.0
61935,preterm_status,term,0.80,R3,paper.fulltext.methods,PMC12222458/methods,"66 of healthy, full-term mother-infant pairs"
61936,sex,female,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!sex:135,sex=Female


## 5. Coverage per field

In [ ]:
cov = pd.read_csv('field_coverage_summary.csv', index_col=0)
cov.frac_infant_samples.sort_values().plot.barh(figsize=(5,4)); plt.xlabel('fraction of infant-scope samples with a value'); plt.tight_layout(); plt.show()

## 6. Get the runs for download (e.g. `fasterq-dump` / ENA FTP)

In [ ]:
runs = pd.read_parquet('runs.parquet')
my_runs = runs[runs.sample_accession.isin(sl.sample_key)]
print(len(my_runs), 'runs')
my_runs[['run_accession','sample_accession','study_accession','library_layout','instrument_model','read_count']].head()

1160 runs


,run_accession,sample_accession,study_accession,library_layout,instrument_model,read_count
9587,ERR12812864,SAMEA115454995,PRJEB74322,PAIRED,Illumina HiSeq 2500,18122268
9595,ERR12813208,SAMEA115455060,PRJEB74322,PAIRED,Illumina HiSeq 2500,32203140
9596,ERR12813219,SAMEA115455027,PRJEB74322,PAIRED,Illumina HiSeq 2500,23039418
9597,ERR12814168,SAMEA115454991,PRJEB74322,PAIRED,Illumina HiSeq 2500,16857338
9598,ERR12814185,SAMEA115455030,PRJEB74322,PAIRED,Illumina HiSeq 2500,23809548
